# Prétraitement du texte

On traite tous les tweets pour garder l'information pertinente pour la classification :
- Lemmatization
- Suppression des stop words et de la ponctuation
- Encodage des emoji, hashtag, nombre,...

In [16]:
import spacy
import contractions
from nltk.tokenize import word_tokenize
import re
import emoji
import pandas as pd

nlp = spacy.load("en_core_web_sm")

## Prétraitement 1

On tokenise puis on analyse chaque token:
- Si c'est un mot important on lemmatize et on garde le token
- Si c'est un nombre, une mention, un hashtag, ... on sauvegarde sa présence dans une variable

In [17]:
url_pattern = r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+'

def replace_contraction(txt):
   return contractions.fix(txt)

def keep_token(t):
    if not t.is_alpha:
        return False
    if t.pos_ not in ['ADJ', 'ADV', 'INTJ', 'NOUN', 'PROPN', 'VERB']:
        return False
    return True
    
def process_entry(x):
    doc = nlp(replace_contraction(x["text"]))
    tokens = []
    has_number = False
    has_money = False
    has_percent = False
    has_url = False
    has_emoji = False
    has_mention = False
    has_hash = False
    for token in doc:
        if keep_token(token):
            tokens.append(token.lemma_.lower())
        has_number |= token.pos_ == 'NUM'
        has_money |= token.ent_type_ == 'MONEY'
        has_percent |= token.pos_ == 'PERCENT'
        has_url |= re.match(url_pattern, token.text) is not None
        has_emoji |= emoji.is_emoji(token.text)
        has_mention |= token.text.startswith("@")
        has_hash |= token.text.startswith("#")

    return {'tokens':tokens, 'has_num': int(has_number), 'has_perc': int(has_percent), 'has_url': int(has_url), 'has_emoji': int(has_emoji), 'has_mention': int(has_mention), 'has_hash': int(has_hash)}    

In [18]:
data = pd.read_csv("scitweets_export.tsv", sep="\t", index_col=0)
data = data.drop(columns=["tweet_id"])

processed = data.apply(process_entry, axis=1)
data["tokens"] = [" ".join(p['tokens']) for p in processed]
data["has_num"] = [p['has_num'] for p in processed]
data["has_perc"] = [p['has_perc'] for p in processed]
data["has_url"] = [p['has_url'] for p in processed]
data["has_emoji"] = [p['has_emoji'] for p in processed]
data["has_mention"] = [p['has_mention'] for p in processed]
data["has_hash"] = [p['has_hash'] for p in processed]

data = data.drop(columns=["text"])

with open("data_pp1.csv", "w") as f:
    f.write("# Preprocessed by saving the presence of each named entity as a feature\n")
    data.to_csv(f, index=False)

## Prétraitement 2 (version alternative)

Comme le prétraitement 1, sauf que pour chaque nombre, pourcentage, mention, emoji ect... On le remplace directement par un identifiant dans le texte : par exemple toute url est remplacé par le mot 'url'.
Ca pourrait améliorer la vectorization car :
- on garde la position dans le tweet du token
- on garde le nombre d'occurence dans le tweet (si il y a plusieurs nombres dans un même tweet par exemple)
- il pourront être traité comme des mots du texte lors de la vectorization, donc leur poids sera plus justement normalizé par tf-idf 

In [19]:
def replace_contraction(txt):
   return contractions.fix(txt)

def remove_tags(txt):
    """ Supprime les # pour traiter les tags comme des mots
    """
    return txt.replace("#", " hashtag ")

def format_mention(txt):
    """ Remplace chaque mention par 'person', car elles représentent principelement des personnes
    """
    expr = r'(?:@[\w_]+)'
    return re.sub(expr, " person ", txt)

def format_links(txt):
    """ Remplace chaque url par 'url' dans le tweets
    """
    expr = r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+'
    return re.sub(expr, " url ", txt)

def format_emoji(txt):
    """ Remplace chaque emoji par 'emoticon' dans le tweets
    """
    new_txt = txt
    for token in reversed(nlp(txt)):
        if emoji.is_emoji(token.text):
            new_txt = new_txt[:token.idx] + " emoticon " + new_txt[token.idx + len(token.text):]
    return new_txt

def format_named_entities(txt):
    """ Remplace chaque entité nommée par son label
    """
    doc = nlp(txt)

    # Build a new string with entity labels instead of entity text
    new_txt = txt
    for ent in reversed(doc.ents):  # reverse so replacements don't mess up indices
        if not "#" in ent.text and ent.label_ in ["CARDINAL", "PERCENT", "MONEY"] :
            if ent.label_ == "CARDINAL":
                lab = ""
            else:
                lab = ent.label_
            new_txt = new_txt[:ent.start_char] + " number " + lab + " " + new_txt[ent.end_char:]
    return new_txt

def format_txt(txt):
    """ Applique toutes les transformation spécifiques
    """
    new_txt = replace_contraction(txt)
    new_txt = format_emoji(format_links(format_mention(remove_tags(new_txt))))
    new_txt = format_named_entities(new_txt)
    return new_txt

def valid_word(t):
    if not t.is_alpha:
        return False
    if t.pos_ not in ['ADJ', 'ADV', 'INTJ', 'NOUN', 'PROPN', 'VERB']:
        return False
    return True

def process_entry_2(x):
    txt = format_txt(x["text"])
    tokens = []
    for token in nlp(txt):
        if valid_word(token):
            tokens.append(token.lemma_.lower())

    return " ".join(tokens)   

In [20]:
data = pd.read_csv("scitweets_export.tsv", sep="\t", index_col=0)

data["tokens"] = data.apply(process_entry_2, axis=1)

data = data.drop(columns=["text"])
data = data.drop(columns=["tweet_id"])

data = data[~data['tokens'].isna()]

with open("data_pp2.csv", "w") as f:
    f.write("# Preprocessed by replacing every named entities by their label, directly in the text\n")
    data.to_csv(f, index=False)